# Transforms

> Data transformations and augmentations for 2D and 3D BioImages

In [ ]:
#| default_exp transforms

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from fastai.vision.all import *
from fastai.data.all import *
from monai.transforms import SpatialCrop, Flip, Rotate90

# Data Augmentation

## Utilities

In [ ]:
# # Define a random tensor
# orig_size = (10, 8)
# rand_tensor = torch.rand(3, *orig_size)  # Random tensor with shape (3, 10, 8)

# # Specify the desired size and top-left corner for cropping
# size = (12, 9)
# tl = (2, 3)

# # Perform cropping
# result_tensor = crop_pad_tensor(rand_tensor, size, tl, orig_size)

# print("Original tensor shape:", rand_tensor.shape)
# print("Cropped tensor shape:", result_tensor.shape)

In [ ]:
#| export

def _process_sz(size, ndim=3):
    if isinstance(size,int): 
        size=(size,)*ndim
    return fastuple(size)

def _get_sz(x):
    if isinstance(x, tuple): x = x[0]
    if not isinstance(x, Tensor): return fastuple(x.size)
    return fastuple(getattr(x, 'img_size', getattr(x, 'sz', (x.shape[1:])))) # maybe it should swap x and y axes 

In [ ]:
# Define a random tensor
orig_size = (10, 8)
rand_tensor = torch.rand(3, *orig_size)  # Random tensor with shape (3, 10, 8)

test_eq((3,)*3,_process_sz(3,ndim=3))
test_eq(orig_size,_get_sz((rand_tensor, rand_tensor)))

## Random Transforms

### Random Crop 2D

Random crop of 2D images

In [ ]:
#| export

class RandCrop2D(RandTransform):
    "Randomly crop an image to `size`"
    split_idx,order = None,1
    def __init__(self, 
        size:int|tuple, # Size to crop to, duplicated if one value is specified
        lazy = False,   # a flag to indicate whether this transform should execute lazily or not. Defaults to False
        **kwargs
    ):
        size = _process_sz(size, ndim=2)
        store_attr()
        super().__init__(**kwargs)

    def before_call(self, 
        b, 
        split_idx:int # Index of the train/valid dataset
    ):
        "Randomly positioning crop if train dataset else center crop"
        self.orig_sz = _get_sz(b)
        if split_idx: self.ctr = (self.orig_sz)//2
        else:
            wd = self.orig_sz[0] - self.size[0]
            hd = self.orig_sz[1] - self.size[1]
            w_rand = (wd, -1) if wd < 0 else (0, wd)
            h_rand = (hd, -1) if hd < 0 else (0, hd)
            self.ctr = fastuple(random.randint(*w_rand)+self.size[0]//2, random.randint(*h_rand)+self.size[1]//2)

    def encodes(self, x):
        return SpatialCrop(roi_center=self.ctr, roi_size=self.size, lazy=self.lazy)(x)

### Random Crop ND

Random crop of n-dimensional images

In [ ]:
#| export

class RandCropND(RandTransform):
    """
    Randomly crops an ND image to a specified size.

    This transform randomly crops an ND image to a specified size during training and performs
    a center crop during validation. It supports both 2D and 3D images and videos, assuming
    the first dimension is the batch dimension.

    Args:
        size (int or tuple): The size to crop the image to. this can have any number of dimensions. 
                             If a single value is provided, it will be duplicated for each spatial 
                             dimension, up to a maximum of 3 dimensions.
        **kwargs: Additional keyword arguments to be passed to the parent class.
    """

    split_idx,order = None,1
        
    def __init__(self, size: int | tuple, # Size to crop to, duplicated if one value is specified
                 lazy = False,            # a flag to indicate whether this transform should execute lazily or not. Defaults to False
                 **kwargs):
        size = _process_sz(size)
        store_attr()
        super().__init__(**kwargs)

    def before_call(self, b, split_idx: int):
        "Randomly position crop if train dataset else center crop"
        self.orig_sz = _get_sz(b)
        if split_idx:
            self.tl = tuple((osz - sz) // 2 for osz, sz in zip(self.orig_sz, self.size))
            self.br = tuple((osz + sz) // 2 for osz, sz in zip(self.orig_sz, self.size))
        else:
            tl = [] # top-left corner
            br = [] # bottom-right corner
            # Calculate top-left and bottom-right corner coordinates for random crop
            for osz, sz in zip(self.orig_sz, self.size):
                w_dif = osz - sz
                if w_dif < 0:
                    w_rand = (0, 0) # No random cropping if input size is smaller than crop size
                    sz = osz # Adjust crop size to match input size
                else:
                    w_rand = (0, w_dif)
                rnd = random.randint(*w_rand)
                tl.append(rnd)
                br.append(rnd + sz)
            self.tl = fastuple(*tl)
            self.br = fastuple(*br)

    def encodes(self, x):
        "Apply spatial crop transformation to the input image."
        return SpatialCrop(roi_start=self.tl, roi_end=self.br, lazy=self.lazy)(x)
    

In [ ]:
# Define a random tensor
orig_size = (65, 65)
rand_tensor = torch.rand(8, *orig_size) 

for i in range(100):
    test_eq((8,64,64),RandCropND((64,64))(rand_tensor).shape)

### Random Crop ND with threshold

Random crop of n-dimensional images assuring that the average intensity of the cropped patch is higher than threshold.

In [ ]:
#| export

class RandCropND_T(RandTransform):
    """
    Randomly crops an ND image to a specified size.

    This transform randomly crops an ND image to a specified size during training and performs
    a center crop during validation. It supports both 2D and 3D images and videos, assuming
    the first dimension is the batch dimension.

    Args:
        size (int or tuple): The size to crop the image to. this can have any number of dimensions. 
                             If a single value is provided, it will be duplicated for each spatial 
                             dimension, up to a maximum of 3 dimensions.
        threshold (float): The minimum average intensity threshold for the cropped patch.
        **kwargs: Additional keyword arguments to be passed to the parent class.
    """

    split_idx, order = None, 1
        
    def __init__(self, size: int | tuple, threshold: float, max_count=5, lazy=False, **kwargs):
        size = _process_sz(size)
        store_attr()
        super().__init__(**kwargs)

    def before_call(self, b, split_idx: int):
        "Randomly position crop if train dataset else center crop"
        self.orig_sz = _get_sz(b)
        if split_idx:
            self.tl = tuple((osz - sz) // 2 for osz, sz in zip(self.orig_sz, self.size))
            self.br = tuple((osz + sz) // 2 for osz, sz in zip(self.orig_sz, self.size))
        else:
            tl = [] # top-left corner
            br = [] # bottom-right corner
            for osz, sz in zip(self.orig_sz, self.size):
                w_dif = osz - sz
                if w_dif < 0:
                    w_rand = (0, 0) # No random cropping if input size is smaller than crop size
                    sz = osz # Adjust crop size to match input size
                else:
                    w_rand = (0, w_dif)
                rnd = random.randint(*w_rand)
                tl.append(rnd)
                br.append(rnd + sz)
            self.tl = fastuple(*tl)
            self.br = fastuple(*br)

    def encodes(self, x):
        "Apply spatial crop transformation to the input image."
        count = 0
        while count < self.max_count:
            # Crop image
            cropped_img = SpatialCrop(roi_start=self.tl, roi_end=self.br, lazy=self.lazy)(x)
            # Check average intensity
            avg_intensity = np.mean(cropped_img)
            if (avg_intensity >= self.threshold):
                break # Accept the crop if intensity is above threshold
            # Choose new random crop
            # print("repeat")
            count += 1
            self.before_call(x, None)
        return cropped_img
    

In [ ]:
# Define a random tensor
orig_size = (65, 65)
rand_tensor = torch.rand(8, *orig_size) 

for i in range(1):
    tst = RandCropND_T((64,64), 1, 2)(rand_tensor)
    test_eq((8,64,64),tst.shape)

### Random Flip

In [ ]:
#| export

class RandFlip(RandTransform):
    """
    Randomly flips an ND image over a specified axis.

    """

    split_idx,order = None,1
        
    def __init__(self, 
                 prob = 0.1,            # Probability of flipping
                 spatial_axis = None,   # Spatial axes along which to flip over. Default is None. The default axis=None will flip over all of the axes of the input array. 
                                        # If axis is negative it counts from the last to the first axis. If axis is a tuple of ints, flipping is performed on all of the axes specified in the tuple.
                 ndim = 2,
                 lazy = False,          # Flag to indicate whether this transform should execute lazily or not. Defaults to False
                 **kwargs):
        store_attr()
        super().__init__(**kwargs)

    def before_call(self, b, split_idx: int):
        if split_idx:
            self.flip = 0
        else:
            self.flip = np.random.choice([0, 1], p=[1-self.prob, self.prob])
        if self.spatial_axis is None:
            self.spatial_axis = np.random.choice(np.arange(self.ndim), size=np.random.randint(1, self.ndim+1), replace=False, p=None)
            
    def encodes(self, x):
        if self.flip:
            return Flip(spatial_axis=self.spatial_axis, lazy=self.lazy)(x)
        else:
            return x

In [ ]:
# Define a random tensor
orig_size = (1,4,4)
rand_tensor = torch.rand(*orig_size) 

print('orig tensor: ', rand_tensor, '\n')

for i in range(3):
    print(RandFlip(prob=.75, spatial_axis=None)(rand_tensor))

orig tensor:  tensor([[[0.1329, 0.4358, 0.5974, 0.0011],
         [0.8778, 0.4527, 0.1575, 0.5419],
         [0.8945, 0.3887, 0.8519, 0.3610],
         [0.3527, 0.5458, 0.0610, 0.7919]]]) 

metatensor([[[0.7919, 0.0610, 0.5458, 0.3527],
         [0.3610, 0.8519, 0.3887, 0.8945],
         [0.5419, 0.1575, 0.4527, 0.8778],
         [0.0011, 0.5974, 0.4358, 0.1329]]])
tensor([[[0.1329, 0.4358, 0.5974, 0.0011],
         [0.8778, 0.4527, 0.1575, 0.5419],
         [0.8945, 0.3887, 0.8519, 0.3610],
         [0.3527, 0.5458, 0.0610, 0.7919]]])
metatensor([[[0.0011, 0.5974, 0.4358, 0.1329],
         [0.5419, 0.1575, 0.4527, 0.8778],
         [0.3610, 0.8519, 0.3887, 0.8945],
         [0.7919, 0.0610, 0.5458, 0.3527]]])


### RandomRot90 

In [ ]:
#| export

class RandRot90(RandTransform):
    """
    Randomly rotate an ND image by 90 degrees in the plane specified by axes.

    """

    split_idx,order = None,1
        
    def __init__(self, 
                 prob = 0.1,            # Probability of rotating
                 max_k = 3,             # Max number of times to rotate by 90 degrees
                 spatial_axes = (0, 1),   # Spatial axes along which to rotate. Default: (0, 1), this is the first two axis in spatial dimensions.
                 ndim = 2,
                 lazy = False,          # Flag to indicate whether this transform should execute lazily or not. Defaults to False
                 **kwargs):
        store_attr()
        super().__init__(**kwargs)

    def before_call(self, b, split_idx: int):
        if split_idx:
            self.rot90 = 0
        else:
            self.rot90 = np.random.choice([0, 1], p=[1-self.prob, self.prob])
            self.k = 1 + np.random.randint(self.max_k)
        # if self.spatial_axes is None:
        #     self.spatial_axes = np.random.choice(np.arange(self.ndim), size=np.random.randint(1, self.ndim+1), replace=False, p=None)
            
    def encodes(self, x):
        if self.rot90:
            return Rotate90(k=self.k, spatial_axes=self.spatial_axes, lazy=self.lazy)(x)
        else:
            return x

In [ ]:
# Define a random tensor
orig_size = (1,4,4)
rand_tensor = torch.rand(*orig_size) 

print('orig tensor: ', rand_tensor, '\n')

for i in range(3):
    print(RandRot90(prob=.75)(rand_tensor))

orig tensor:  tensor([[[0.8310, 0.7337, 0.4620, 0.6483],
         [0.2909, 0.9434, 0.2867, 0.2595],
         [0.0954, 0.1127, 0.1863, 0.6641],
         [0.4874, 0.4285, 0.0029, 0.2368]]]) 

metatensor([[[0.2368, 0.0029, 0.4285, 0.4874],
         [0.6641, 0.1863, 0.1127, 0.0954],
         [0.2595, 0.2867, 0.9434, 0.2909],
         [0.6483, 0.4620, 0.7337, 0.8310]]])
metatensor([[[0.2368, 0.0029, 0.4285, 0.4874],
         [0.6641, 0.1863, 0.1127, 0.0954],
         [0.2595, 0.2867, 0.9434, 0.2909],
         [0.6483, 0.4620, 0.7337, 0.8310]]])
tensor([[[0.8310, 0.7337, 0.4620, 0.6483],
         [0.2909, 0.9434, 0.2867, 0.2595],
         [0.0954, 0.1127, 0.1863, 0.6641],
         [0.4874, 0.4285, 0.0029, 0.2368]]])


---

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()